In [25]:
import pandas as pd
from ydata_profiling import ProfileReport
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from autogluon.tabular import TabularPredictor
import plotly.express as px
from pprint import pprint
from autogluon.features.generators import AutoMLPipelineFeatureGenerator

In [11]:
csv_path = Path.cwd().parents[1] / "exports" / "survey_2025_scalar_columns.csv"
df = pd.read_csv(csv_path, low_memory=False)

In [12]:
pd.options.display.max_columns = 500

df

,ResponseId,MainBranch,Age,EdLevel,Employment,WorkExp,LearnCodeChoose,LearnCodeAI,YearsCode,DevType,OrgSize,ICorPM,RemoteWork,PurchaseInfluence,TechEndorseIntro,Industry,AIThreat,NewRole,ToolCountWork,ToolCountPersonal,Country,Currency,CompTotal,LanguageChoice,DatabaseChoice,PlatformChoice,WebframeChoice,DevEnvsChoice,AIModelsChoice,SOAccount,SOVisitFreq,SODuration,SOPartFreq,SOComm,SOFriction,AISelect,AISent,AIAcc,AIComplex,AIAgents,AIAgentChange,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,8.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",14.0,"Developer, mobile",20 to 99 employees,People manager,Remote,"Yes, I influenced the purchase of a substantial addition to the tech stack",Work,Fintech,I'm not sure,I have neither consider or transitioned into a new career or industry,7.0,3.0,Ukraine,EUR European Euro,52800.0,Yes,Yes,Yes,No,Yes,Yes,Yes,A few times per week,Between 5 and 10 years,I have never participated in Q&A on Stack Overflow,Neutral,"Rarely, almost never","Yes, I use AI tools monthly or infrequently",Indifferent,Neither trust nor distrust,Bad at handling complex tasks,"Yes, I use AI agents at work monthly or infrequently",Not at all or minimally,61256.0,10.0
1,2,I am a developer by profession,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,2.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",10.0,"Developer, back-end",500 to 999 employees,Individual contributor,"Hybrid (some in-person, leans heavy to flexibility)",No,Personal Project,Retail and Consumer Services,I'm not sure,I have transitioned into a new career and/or industry voluntarily,6.0,5.0,Netherlands,EUR European Euro,90000.0,Yes,Yes,Yes,Yes,Yes,Yes,Not sure/can't remember,Multiple times per day,Between 10 and 15 years,"Infrequently, less than once per year","Yes, somewhat",About half of the time,"Yes, I use AI tools weekly",Indifferent,Neither trust nor distrust,Bad at handling complex tasks,"No, and I don't plan to",Not at all or minimally,104413.0,9.0
2,3,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-employed",10.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",12.0,"Developer, front-end",NaN,NaN,NaN,No,Work,Software Development,No,I have transitioned into a new career and/or industry involuntarily,3.0,3.0,Ukraine,UAH Ukrainian hryvnia,2214000.0,Yes,Yes,Yes,Yes,Yes,Yes,Not sure/can't remember,A few times per week,Between 5 and 10 years,"Infrequently, less than once per year",Neutral,About half of the time,"Yes, I use AI tools daily",Favorable,Somewhat trust,Neither good or bad at handling complex tasks,"Yes, I use AI agents at work weekly","Yes, somewhat",53061.0,8.0
3,4,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,4.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",5.0,"Developer, back-end","10,000 or more employees",Individual contributor,Remote,No,Personal Project,Retail and Consumer Services,No,I have neither consider or transitioned into a new career or industry,NaN,NaN,Ukraine,EUR European Euro,31200.0,Yes,No,Yes,Yes,No,No,Yes,A few times per month or weekly,Between 3 and 5 years,I have never participated in Q&A on Stack Overflow,Neutral,"Rarely, almost never","Yes, I use AI tools weekly",Favorable,Somewhat trust,Bad at handling complex tasks,"Yes, I use AI agents at work monthly or 

In [ ]:
# Auto EDA
# profile = ProfileReport(df, title="Stack Overflow Developer Survey 2025 - Public Results (Scalars)")
# profile.to_file("so_survey_2025_profile_scalars_report.html")
# profile.to_file("so_survey_2025_profile_scalars_report.json")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 227.48it/s]


### Baseline with AutoGluon Tabular: predict `ConvertedCompYearly`

**Goal:** A strong default model with minimal manual feature engineering.

**How AutoGluon Tabular works:** We pass one dataframe with a **label** column. On `fit`, it (1) runs a **feature pipeline** (dtype inference, categorical encoding, and optional text/datetime handling), (2) **fits many models** (e.g. gradient boosting, random forest, neural nets) using **internal** train/validation splits of the training frame you pass in, (3) picks and often **ensembles** winners by validation score. It does **not** remove **label leakage** for us, anything that is effectively part of the target definition must be dropped **before** `fit`.

- **Target filter** — `ConvertedCompYearly` is continuous compensation in USD. Rows with missing or non-positive targets are dropped so supervised training and metrics are well-defined.
- **Drop `LEAK_COLS`** — `ConvertedCompYearly` is derived from `CompTotal`, `Currency`, and Stack Overflow’s conversion rules. Using those as features would **leak** the answer and make test scores meaningless. `ResponseId` is only an identifier, not a causal input.
- **`train_test_split` (80/20)** — This is an **outer** test set: never passed to `fit`, used only for final evaluation. AutoGluon will **again** split `train_data` inside `fit` for model selection and stacking; that inner split is not a replacement for this holdout.


In [ ]:
# Regression label: annual compensation in USD
TARGET = "ConvertedCompYearly"
# Do not use as features — CompTotal/Currency are inputs to the same conversion as the label; ResponseId is not predictive.
LEAK_COLS = ["ResponseId", "CompTotal", "Currency"]

# Supervised rows only: need a real positive salary for training and sensible metrics.
df = df[df[TARGET].notna() & (df[TARGET] > 0)].reset_index(drop=True)
df = df.drop(columns=[c for c in LEAK_COLS if c in df.columns])

# Outer holdout for honest test metrics. TabularPredictor.fit() still splits this train set internally.
train_data, test_data = train_test_split(
    df, test_size=0.2, random_state=42, shuffle=True
)

print(f"Rows: {len(df):,}  |  Train: {len(train_data):,}  |  Test: {len(test_data):,}")
train_data.head()

Rows: 23,947  |  Train: 19,157  |  Test: 4,790


,MainBranch,Age,EdLevel,Employment,WorkExp,LearnCodeChoose,LearnCodeAI,YearsCode,DevType,OrgSize,ICorPM,RemoteWork,PurchaseInfluence,TechEndorseIntro,Industry,AIThreat,NewRole,ToolCountWork,ToolCountPersonal,Country,LanguageChoice,DatabaseChoice,PlatformChoice,WebframeChoice,DevEnvsChoice,AIModelsChoice,SOAccount,SOVisitFreq,SODuration,SOPartFreq,SOComm,SOFriction,AISelect,AISent,AIAcc,AIComplex,AIAgents,AIAgentChange,ConvertedCompYearly,JobSat
18879,I am a developer by profession,35-44 years old,"Professional degree (JD, MD, Ph.D, Ed.D, etc.)",Employed,3.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Yes, I learned how to use AI-enabled tools required for my job or to benefit my career",5.0,Data engineer,100 to 499 employees,Individual contributor,"Hybrid (some in-person, leans heavy to flexibility)",No,Work,Software Development,No,I have transitioned into a new career and/or industry voluntarily,3.0,0.0,United Kingdom of Great Britain and Northern Ireland,Yes,Yes,Yes,No,Yes,Yes,Yes,A few times per month or weekly,Between 3 and 5 years,"Infrequently, less than once per year","No, not really","Rarely, almost never","Yes, I use AI tools daily",Very favorable,Somewhat trust,I don't use AI tools for complex tasks / I don't know,"No, I use AI exclusively in copilot/autocomplete mode",Not at all or minimally,59962.0,10.0
3162,I am a developer by profession,25-34 years old,"Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)",Employed,6.0,"No, I am not new to coding and did not learn new coding techniques or programming languages","No, I didn't spend time learning in the past year",6.0,"Developer, front-end",20 to 99 employees,Individual contributor,Remote,No,Personal Project,Higher Education,Yes,I have somewhat considered changing my career and/or the industry I work in,13.0,5.0,Germany,Yes,Yes,Yes,Yes,Yes,NaN,No,A few times per month or weekly,Between 5 and 10 years,I have never participated in Q&A on Stack Overflow,"No, not at all","Rarely, almost never","Yes, I use AI tools daily",Favorable,Somewhat distrust,Very poor at handling complex tasks,"Yes, I use AI agents at work monthly or infrequently","Yes, to a great extent",81210.0,8.0
15568,I am a developer by profession,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,13.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",19.0,"Developer, back-end","10,000 or more employees",Individual contributor,"Your choice (very flexible, you can come in when you want or just as needed)",No,Work,Fintech,No,I have somewhat considered changing my career and/or the industry I work in,10.0,NaN,Romania,Yes,Yes,Yes,No,Yes,Yes,Yes,A few times per week,Between 10 and 15 years,Less than once every 2 - 3 months,"Yes, somewhat","Rarely, almost never","Yes, I use AI tools monthly or infrequently",Favorable,Neither trust nor distrust,Neither good or bad at handling complex tasks,"No, I use AI exclusively in copilot/autocomplete mode","Yes, somewhat",88171.0,8.0
1731,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,19.0,"No, I am not new to coding and did not learn new coding techniques or programming languages","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",19.0,"Developer, full-stack",Less than 20 employees,Individual contributor,Remote,No,Work,Software Development,I'm not sure,I have somewhat considered changing my career and/or the industry I work in,20.0,0.0,United States of America,Yes,Yes,Yes,Yes,Yes,No,Yes,A few times per month or weekly,Between 10 and 15 years,I have never participated in Q&A on Stack Overflow,"No, not at all","Rarely, almost never","Yes, I use AI tools daily",Favorable,Somewhat trust,"Good, but not great at handling complex tasks","No, but I plan to","Yes, so

In [26]:
X = train_data.drop(columns=[TARGET])  # or your label name
fg = AutoMLPipelineFeatureGenerator()
X_processed = fg.fit_transform(X)
X_processed.head()

Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    7913.76 MB
	Train Data (Original)  Memory Usage: 47.23 MB (0.6% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', [])  :  5 | ['WorkExp', 'YearsCode', 'ToolCountWork', 'ToolCountPersonal', 'JobSat']
		('object', []) : 34 | ['MainBranch', 'Age', 'EdLevel', 'Employment', 'LearnCodeChoose', ...]
	Types of features in processed data (raw dtype, sp

,WorkExp,YearsCode,ToolCountWork,ToolCountPersonal,JobSat,MainBranch,Age,EdLevel,Employment,LearnCodeChoose,LearnCodeAI,DevType,OrgSize,ICorPM,RemoteWork,PurchaseInfluence,TechEndorseIntro,Industry,AIThreat,NewRole,Country,LanguageChoice,DatabaseChoice,PlatformChoice,WebframeChoice,DevEnvsChoice,AIModelsChoice,SOAccount,SOVisitFreq,SODuration,SOPartFreq,SOComm,SOFriction,AISelect,AISent,AIAcc,AIComplex,AIAgents,AIAgentChange
18879,3.0,5.0,3.0,0.0,10.0,0,2,6,0,3,5,6,3,1,1,1,3,14,2,5,132,2,2,2,1,2,2,3,1,3,5,3,5,3,5,5,3,1,3
3162,6.0,6.0,13.0,5.0,8.0,0,1,7,0,1,1,16,4,1,4,1,1,7,3,2,42,2,2,2,2,2,0,1,1,4,4,2,5,3,1,4,5,5,5
15568,13.0,19.0,10.0,NaN,8.0,0,2,3,0,3,4,13,2,1,5,1,3,4,2,2,104,2,2,2,1,2,2,3,2,2,6,6,5,4,1,3,4,1,4
1731,19.0,19.0,20.0,0.0,9.0,0,2,2,0,1,4,17,9,1,4,1,3,14,1,2,134,2,2,2,2,2,1,3,1,2,4,2,5,3,1,5,2,3,4
16899,1.0,2.0,5.0,2.0,NaN,1,0,2,0,3,4,7,6,1,2,1,1,9,3,3,51,2,2,1,1,2,1,3,1,6,4,6,5,2,5,5,3,3,4


In [30]:
px.histogram(X_processed['WorkExp'])

In [16]:
problem_type = "regression"
metric = "mean_absolute_error"
time_in_minutes = 10
time_limit_in_seconds = time_in_minutes * 60
presets = "best_quality"

predictor = TabularPredictor(
    label=TARGET,
    problem_type=problem_type,
    eval_metric=metric,
).fit(
    train_data=train_data,
    time_limit=time_limit_in_seconds,
    presets=presets,
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260414_160116"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.5
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.3.0: Wed Jan 28 20:54:46 PST 2026; root:xnu-12377.91.3~2/RELEASE_ARM64_T6000
CPU Count:          10
Pytorch Version:    2.9.1
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       6.90 GB / 32.00 GB (21.6%)
Disk Space Avail:   90.66 GB / 926.35 GB (9.8%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack

In [ ]:
# Load a specific model
# model_name = "ag-20260411_214704"
# PREDICTOR_DIR = Path.cwd() / "AutogluonModels" / model_name
# predictor = TabularPredictor.load(str(PREDICTOR_DIR.resolve()))

In [17]:
predictor.leaderboard(test_data)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L3,-40998.822516,-42184.207380,mean_absolute_error,2.272561,2.268790,385.341789,0.001665,0.000555,0.058814,3,True,12
1,CatBoost_BAG_L2,-41021.553592,-42201.266399,mean_absolute_error,2.270896,2.268235,385.282975,0.079172,0.152113,103.522172,2,True,9
2,WeightedEnsemble_L2,-41107.096895,-42315.538961,mean_absolute_error,0.550418,0.355728,253.844382,0.001638,0.000442,0.048580,2,True,7
3,CatBoost_BAG_L1,-41176.018443,-42393.893980,mean_absolute_error,0.205479,0.177360,229.936886,0.205479,0.177360,229.936886,1,True,2
4,CatBoost_r177_BAG_L1,-41920.638251,-43156.812047,mean_absolute_error,0.029332,0.105560,2.884699,0.029332,0.105560,2.884699,1,True,6
5,NeuralNetTorch_BAG_L1,-44065.934311,-46798.599044,mean_absolute_error,0.343301,0.177926,23.858916,0.343301,0.177926,23.858916,1,True,5
6,NeuralNetFastAI_BAG_L1,-48499.372616,-55925.827745,mean_absolute_error,0.436496,0.251136,14.832108,0.436496,0.251136,14.832108,1,True,4
7,NeuralNetFastAI_BAG_L2,-48911.938215,-50307.325340,mean_absolute_error,2.648738,2.351177,295.762796,0.457014,0.235056,14.001993,2,True,11
8,RandomForestMSE_BAG_L2,-51178.595717,-58102.654064,mean_absolute_error,2.485001,2.836947,295.445206,0.293277,0.720825,13.684403,2,True,8
9,ExtraTreesMSE_BAG_L2,-54966.319962,-58304.702928,mean_absolute_error,2.427948,2.888604,285.828236,0.236225,0.772482,4.067433,2,True,10


In [22]:
predictor.evaluate(test_data)


{'mean_absolute_error': -40998.8225163629,
 'root_mean_squared_error': np.float64(-231776.2931159863),
 'mean_squared_error': -53720250050.5876,
 'r2': 0.04946239768736671,
 'pearsonr': 0.23479075157718104,
 'median_absolute_error': -17194.935546875}

In [21]:
predictor.model_best

'WeightedEnsemble_L3'

In [23]:
predictor.feature_importance(test_data)

Computing feature importance via permutation shuffling for 39 features using 4790 rows with 5 shuffle sets...
	300.97s	= Expected runtime (60.19s per shuffle set)
	152.33s	= Actual runtime (Completed 5 of 5 shuffle sets)


,importance,stddev,p_value,n,p99_high,p99_low
Country,24364.608881,324.217534,3.761724e-09,5,25032.177265,23697.040497
WorkExp,4017.417273,100.829964,4.757593e-08,5,4225.027561,3809.806984
OrgSize,1770.163181,152.446338,6.536022e-06,5,2084.052297,1456.274064
DevType,1222.945905,49.561489,3.229827e-07,5,1324.993696,1120.898115
Industry,821.077637,65.793022,4.905178e-06,5,956.546379,685.608895
Age,820.545173,110.627401,3.870519e-05,5,1048.328521,592.761825
YearsCode,587.897890,111.498074,1.480801e-04,5,817.473966,358.321813
Employment,517.900592,45.416709,7.024546e-06,5,611.414224,424.386961
RemoteWork,385.788589,36.683502,9.692802e-06,5,461.320427,310.256751
EdLevel,311.768129,49.093777,7.140626e-05,5,412.852894,210.683364


In [24]:
pprint(predictor.feature_metadata.to_dict())

{'AIAcc': ('category', ()),
 'AIAgentChange': ('category', ()),
 'AIAgents': ('category', ()),
 'AIComplex': ('category', ()),
 'AIModelsChoice': ('category', ()),
 'AISelect': ('category', ()),
 'AISent': ('category', ()),
 'AIThreat': ('category', ()),
 'Age': ('category', ()),
 'Country': ('category', ()),
 'DatabaseChoice': ('category', ()),
 'DevEnvsChoice': ('category', ()),
 'DevType': ('category', ()),
 'EdLevel': ('category', ()),
 'Employment': ('category', ()),
 'ICorPM': ('category', ()),
 'Industry': ('category', ()),
 'JobSat': ('float', ()),
 'LanguageChoice': ('category', ()),
 'LearnCodeAI': ('category', ()),
 'LearnCodeChoose': ('category', ()),
 'MainBranch': ('category', ()),
 'NewRole': ('category', ()),
 'OrgSize': ('category', ()),
 'PlatformChoice': ('category', ()),
 'PurchaseInfluence': ('category', ()),
 'RemoteWork': ('category', ()),
 'SOAccount': ('category', ()),
 'SOComm': ('category', ()),
 'SODuration': ('category', ()),
 'SOFriction': ('category', ())